# RareSight — EDA & Results
Dataset analysis, class distribution, and evaluation results.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np

plt.rcParams.update({'figure.dpi': 120, 'font.family': 'DejaVu Sans', 'axes.spines.top': False, 'axes.spines.right': False})

## 1. Dataset class distribution

In [ ]:
DATASETS = {
    'ISIC 2019': {
        'Melanocytic Nevi': 12875, 'Melanoma': 4522, 'BCC': 3323,
        'Benign Keratosis': 2624, 'Actinic Keratosis': 867, 'SCC': 628,
        'Vascular Lesion*': 253, 'Dermatofibroma*': 239,
    },
    'HAM10000': {
        'Melanocytic Nevi': 6705, 'Melanoma': 1113, 'BKL': 1099,
        'BCC': 514, 'AKIEC': 327, 'Vascular*': 142, 'DF*': 115,
    },
    'PAD-UFES-20': {
        'BCC': 845, 'ACK': 730, 'SEK': 251, 'NEV': 244, 'SCC': 176, 'MEL*': 52,
    },
}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

COLORS = ['#1D9E75', '#D85A30', '#378ADD', '#7F77DD', '#BA7517', '#639922', '#D4537E', '#E24B4A']

for ax, (name, data) in zip(axes, DATASETS.items()):
    classes = list(data.keys())
    counts  = list(data.values())
    colors  = [COLORS[i % len(COLORS)] for i in range(len(classes))]
    alphas  = [0.65 if '*' not in c else 1.0 for c in classes]
    
    bars = ax.bar(classes, counts, color=colors)
    for bar, alpha in zip(bars, alphas):
        bar.set_alpha(alpha)
    
    ax.set_title(f'{name}\n({sum(counts):,} samples)', fontsize=12, fontweight='normal')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=35, labelsize=9)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))

rare_patch = mpatches.Patch(color='#D4537E', label='Rare class (full opacity)')
fig.legend(handles=[rare_patch], loc='lower center', ncol=1, fontsize=10)
plt.tight_layout()
plt.savefig('../outputs/dataset_distribution.png', bbox_inches='tight')
plt.show()

## 2. Class imbalance ratio

In [ ]:
isic = DATASETS['ISIC 2019']
majority = max(isic.values())
print('Imbalance ratio (majority / minority):')
for cls, cnt in sorted(isic.items(), key=lambda x: x[1]):
    ratio = majority / cnt
    print(f'  {cls:<25} {cnt:>6,}   ratio: {ratio:.1f}x')

## 3. MAE reconstruction visualisation (run after Stage 1)

In [ ]:
# Uncomment after Stage 1 training is complete
# import torch
# from raresight.models.mae import MaskedAutoencoder
# from raresight.data.dataset import build_val_transform
# from PIL import Image
# import numpy as np
#
# cfg = {  # match your config
#     'img_size': 224, 'patch_size': 16, 'mask_ratio': 0.75,
#     'encoder': {'embed_dim': 768, 'depth': 12, 'num_heads': 12},
#     'decoder': {'embed_dim': 512, 'depth': 8, 'num_heads': 16},
# }
# model = MaskedAutoencoder(cfg)
# ckpt = torch.load('../checkpoints/stage1_mae_best.pth', map_location='cpu')
# model.load_state_dict(ckpt['model_state_dict'])
# model.eval()
# # ... visualise masked vs reconstructed patches
print('Uncomment after Stage 1 training.')

## 4. Load and display evaluation results

In [ ]:
import json, os

ablation_path = '../outputs/evaluation/ablation_results.json'
if os.path.exists(ablation_path):
    with open(ablation_path) as f:
        ablation = json.load(f)
    
    labels = {'full': 'Multimodal (image + clinical)', 'image': 'Image only', 'clinical': 'Clinical only'}
    metrics = ['macro_auc', 'balanced_acc', 'macro_f1']
    
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, metric in zip(axes, metrics):
        vals   = [ablation[mode][metric] for mode in ['image', 'clinical', 'full']]
        xlabs  = ['Image only', 'Clinical only', 'Multimodal']
        colors = ['#B5D4F4', '#C0DD97', '#1D9E75']
        bars   = ax.bar(xlabs, vals, color=colors)
        ax.set_ylim(0, 1)
        ax.set_title(metric.replace('_', ' ').title(), fontsize=11)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.3f}', ha='center', fontsize=10)
        ax.tick_params(axis='x', rotation=15)
    
    plt.suptitle('Ablation Study: Modality Contribution', fontsize=13)
    plt.tight_layout()
    plt.savefig('../outputs/ablation_chart.png', bbox_inches='tight')
    plt.show()
else:
    print('Run scripts/evaluate.py first to generate ablation results.')

## 5. Literature comparison

In [ ]:
lit_path = '../outputs/evaluation/literature_comparison.csv'
if os.path.exists(lit_path):
    df = pd.read_csv(lit_path)
    print(df.to_string(index=False))
else:
    print('Run scripts/evaluate.py first.')